In [ ]:

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# Создаём состояние финансового агента
class FinanceState(TypedDict):
    income: float
    expenses: float
    savings: float
    response: str


In [ ]:

# Создаём функцию для расчёта накоплений
def calculate_savings(state: FinanceState):
    income = state["income"]
    expenses = state["expenses"]

    savings = income - expenses

    return {"savings": savings}


In [ ]:

# Создаём функцию для финансового ответа
def generate_response(state: FinanceState):
    savings = state["savings"]

    if savings > 0:
        response = f"You can save {savings:,.0f} KZT per month."
    elif savings == 0:
        response = "Your income is equal to your expenses."
    else:
        response = f"Your expenses exceed your income by {abs(savings):,.0f} KZT."

    return {"response": response}


In [ ]:

# Создаём граф финансового агента
graph = StateGraph(FinanceState)

# Добавляем два узла
graph.add_node("calculate_savings", calculate_savings)
graph.add_node("generate_response", generate_response)

# Соединяем узлы
graph.add_edge(START, "calculate_savings")
graph.add_edge("calculate_savings", "generate_response")
graph.add_edge("generate_response", END)

# Компилируем граф
app = graph.compile()

print("Finance Agent graph created successfully!")


In [ ]:

# Вводим финансовые данные пользователя
user_data = {
    "income": 300000,
    "expenses": 220000,
    "savings": 0,
    "response": ""
}

# Запускаем граф
result = app.invoke(user_data)

# Выводим результат
print(result["response"])


In [ ]:

# Создаём три финансовых сценария
test_cases = [
    {"income": 300000, "expenses": 200000},
    {"income": 250000, "expenses": 250000},
    {"income": 180000, "expenses": 220000}
]

# Проверяем работу графа
for case in test_cases:
    result = app.invoke({
        "income": case["income"],
        "expenses": case["expenses"],
        "savings": 0,
        "response": ""
    })

    print(result["response"])

In [ ]:

# Определяем финансовую ситуацию
def check_budget(state: FinanceState):
    savings = state["savings"]

    if savings > 0:
        return "positive"
    elif savings == 0:
        return "balanced"
    else:
        return "negative"


In [ ]:

# Ответ при положительном остатке
def positive_budget(state: FinanceState):
    savings = state["savings"]
    return {
        "response": f"You can save {savings:,.0f} KZT per month."
    }


# Ответ при сбалансированном бюджете
def balanced_budget(state: FinanceState):
    return {
        "response": "Your income is equal to your expenses."
    }


# Ответ при превышении расходов
def negative_budget(state: FinanceState):
    debt = abs(state["savings"])
    return {
        "response": f"Your expenses exceed your income by {debt:,.0f} KZT."
    }


In [ ]:

# Создаём новый граф
graph2 = StateGraph(FinanceState)

# Добавляем узлы
graph2.add_node("calculate_savings", calculate_savings)
graph2.add_node("positive_budget", positive_budget)
graph2.add_node("balanced_budget", balanced_budget)
graph2.add_node("negative_budget", negative_budget)

# Начинаем с расчёта накоплений
graph2.add_edge(START, "calculate_savings")

# Добавляем условные переходы
graph2.add_conditional_edges(
    "calculate_savings",
    check_budget,
    {
        "positive": "positive_budget",
        "balanced": "balanced_budget",
        "negative": "negative_budget"
    }
)

# Завершаем каждый путь
graph2.add_edge("positive_budget", END)
graph2.add_edge("balanced_budget", END)
graph2.add_edge("negative_budget", END)

# Компилируем граф
app2 = graph2.compile()

print("Conditional LangGraph created successfully!")


In [ ]:

# Создаём три финансовых сценария
test_cases = [
    {"income": 300000, "expenses": 200000},
    {"income": 250000, "expenses": 250000},
    {"income": 180000, "expenses": 220000}
]

# Проверяем новый граф
for case in test_cases:
    result = app2.invoke({
        "income": case["income"],
        "expenses": case["expenses"],
        "savings": 0,
        "response": ""
    })

    print(result["response"])


In [ ]:

from IPython.display import Image, display

# Показываем структуру финансового графа
display(Image(app2.get_graph().draw_mermaid_png()))
